In [1]:
import glob
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pyproj import Proj
import xarray
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.feature as cf
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

from global_land_mask import globe

import seaborn as sns

cmap_white_yellow_red = mcolors.LinearSegmentedColormap.from_list(
    "white_yellow_red",
    [
        (0.0, "#FFFFFF"),   # white
        (0.4, "#FFF7BC"),   # pale yellow
        (0.7, "#FEC44F"),   # warm yellow-orange
        (1.0, "#D7301F"),   # deep red
    ]
)


In [ ]:
qa = "aerdt_aod_qa3"
FILE_AHI_agg = f"AGGR_HOURLY_2019236_273_0_25deg_camp2ex_{qa}_FINAL_peat_0.14BA.nc"
C_AHI_agg = xarray.open_dataset(FILE_AHI_agg)
C_AHI_agg

<xarray.Dataset>
Dimensions:    (longitude: 186, latitude: 158)
Coordinates:
  * longitude  (longitude) float64 93.94 94.19 94.44 94.69 ... 139.7 140.0 140.2
  * latitude   (latitude) float64 -19.15 -18.9 -18.65 -18.4 ... 19.65 19.9 20.15
Data variables:
    aod_Mean   (latitude, longitude) float64 ...

In [ ]:
FILE_MUSICAv0_agg = f"AGGR_HOURLY_2019236_273_0_25deg_camp2ex_MUSICAv0_{qa}_FINAL_peat_0.14BA.nc"
C_MUSICAv0_agg = xarray.open_dataset(FILE_MUSICAv0_agg)
C_MUSICAv0_agg 


<xarray.Dataset>
Dimensions:  (lat: 158, lon: 186)
Coordinates:
  * lat      (lat) float32 -19.15 -18.9 -18.65 -18.4 ... 19.4 19.65 19.9 20.15
  * lon      (lon) float32 93.94 94.19 94.44 94.69 ... 139.5 139.7 140.0 140.2
Data variables:
    AODVIS   (lat, lon) float32 ...

In [ ]:
FILE_MUSICAv0_agg_ctrl = f"AGGR_HOURLY_2019236_273_0_25deg_camp2ex_MUSICAv0_{qa}.nc"
C_MUSICAv0_agg_ctrl = xarray.open_dataset(FILE_MUSICAv0_agg_ctrl)
C_MUSICAv0_agg_ctrl


<xarray.Dataset>
Dimensions:  (lat: 158, lon: 186)
Coordinates:
  * lat      (lat) float32 -19.15 -18.9 -18.65 -18.4 ... 19.4 19.65 19.9 20.15
  * lon      (lon) float32 93.94 94.19 94.44 94.69 ... 139.5 139.7 140.0 140.2
Data variables:
    AODVIS   (lat, lon) float32 ...

In [6]:
mean_aod = C_AHI_agg['aod_Mean']
mean_aod_model = C_MUSICAv0_agg['AODVIS']
mean_aod_model_ctrl = C_MUSICAv0_agg_ctrl['AODVIS']

In [7]:
# Ensure NaNs in mean_aod match NaNs in mean_aod_model using a for loop
for latitude in mean_aod.latitude:
    for longitude in mean_aod.longitude:
        value = mean_aod.sel(latitude=latitude, longitude=longitude)
        if (value < 0):
            mean_aod.loc[dict(latitude=latitude, longitude=longitude)] = np.nan
            mean_aod_model.loc[dict(lat=latitude, lon=longitude)] = np.nan
            mean_aod_model_ctrl.loc[dict(lat=latitude, lon=longitude)] = np.nan
        if (np.isnan(value)):
            mean_aod_model.loc[dict(lat=latitude, lon=longitude)] = np.nan
            mean_aod_model_ctrl.loc[dict(lat=latitude, lon=longitude)] = np.nan

In [ ]:
ds_wind_mean = xarray.open_dataset("MUSICAv0_control_SEAcut_25km_2019-08-24_2019-10-05_mean.nc")

In [ ]:
ds_wind_mean_exp01 = xarray.open_dataset("MUSICAv0_exp01_SEAcut_25km_2019-08-24_2019-10-05_mean.nc")

/home/svisaga/miniconda3/envs/cmp/lib/python3.11/site-packages/xarray/core/indexing.py:1228: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/home/svisaga/miniconda3/envs/cmp/lib/python3.11/site-packages/xarray/core/indexing.py:1228: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


In [ ]:
vmax_ = 1.5
longitude_min = 94
longitude_max =140
latitude_min = -15  # Adjust as necessary
latitude_max = 20   # Adjust as necessary

fig = plt.figure(figsize=(10, 8))  # Adjust the figure size for better layout
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1,1],wspace=0.1)  # Define 1 row, 2 columns

pla_proj = ccrs.PlateCarree()
ax1 = fig.add_subplot(gs[0], projection=pla_proj)
ax1.set_extent([longitude_min, longitude_max, latitude_min, latitude_max], crs=ccrs.PlateCarree())

mean_aod.plot.pcolormesh(vmin=0,vmax=vmax_,cmap='YlOrRd', add_colorbar=False)
ahi_mean = mean_aod.mean().item()

plt.title(f'Observation AOD\nHimawari AHI Dark Target\nAOD mean = {ahi_mean:.2f} ',fontsize=10)

gl = ax1.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.xlines = False
gl.ylines = False
gl.right_labels = False
gl.top_labels = False

ax1.coastlines(resolution='50m', color='black', linewidth=1)
ax1.text(-0.2, 1.3, 'a)', transform=ax1.transAxes,
         fontsize=10, va='top', ha='left')

ax2 = fig.add_subplot(gs[1], projection=pla_proj)
ax2.set_extent([longitude_min, longitude_max, latitude_min, latitude_max], crs=ccrs.PlateCarree())

ref = mean_aod_model_ctrl.plot.pcolormesh(vmin=0,vmax=vmax_,cmap='YlOrRd', add_colorbar=False)
musicav0_mean = mean_aod_model_ctrl.mean().item()
ds_wind_mean.plot.streamplot('lon','lat','U','V',color='darkblue',linewidth=0.25,density=1, ax=ax2, zorder=1)

plt.title(f'MUSICAv0 Control AOD\n\nAOD mean= {musicav0_mean:.2f}',fontsize=10)

gl = ax2.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.xlines = False
gl.ylines = False
gl.right_labels = False
gl.left_labels = False
gl.top_labels = False

ax2.coastlines(resolution='50m', color='black', linewidth=1)
ax2.text(-0.07, 1.3, 'b)', transform=ax2.transAxes,
         fontsize=10, va='top', ha='left')

ax2 = fig.add_subplot(gs[2], projection=pla_proj)
ax2.set_extent([longitude_min, longitude_max, latitude_min, latitude_max], crs=ccrs.PlateCarree())

ref = mean_aod_model.plot.pcolormesh(vmin=0,vmax=vmax_,cmap='YlOrRd', add_colorbar=False)
musicav0_mean = mean_aod_model.mean().item()
ds_wind_mean_exp01.plot.streamplot('lon','lat','U','V',color='darkblue',linewidth=0.25,density=1, ax=ax2, zorder=1)

plt.title(f'MUSICAv0 Exp01 AOD\n\nAOD mean= {musicav0_mean:.2f}',fontsize=10)
#11S to 10N, 94-127 E 
gl = ax2.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.xlines = False
gl.ylines = False
gl.right_labels = False
gl.left_labels = False
gl.top_labels = False

ax2.coastlines(resolution='50m', color='black', linewidth=1)
ax2.text(-0.07, 1.3, 'c)', transform=ax2.transAxes,
         fontsize=10, va='top', ha='left')

# Create a common colorbar for both plots
cbar_ax = fig.add_axes([0.2, 0.3, 0.6, 0.02])  # Adjust position as needed
plt.colorbar(ref, cax=cbar_ax, orientation='horizontal')
plt.title(f'AOD')
plt.show()